In [ ]:
## Nikolay Vorontsov, 23.11.2024, Prompting Dataset with hallucinations
## required files: prompts_for_words_nick.env
##                 mushroom.en-val.v2.jsonl

In [ ]:
#INSTALL DEPENDENCIES

!pip install google-generativeai

In [ ]:
# IMPORT LIBRARIES

import configparser
import google.generativeai as genai
import json

from google.colab import userdata
from google.colab import drive

import random

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# DEFINE VARIABLES

Your_API_Key = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=Your_API_Key)

model = genai.GenerativeModel(model_name="gemini-1.5-flash")
prompts = configparser.ConfigParser()

prompts.read('prompt_for_words_nick.env')

output_file="gemini_qa-pairs_with_words_7.1.2025.jsonl"

validation_set = 'mushroom.en-val.v2.jsonl'

In [ ]:
#load random samples from the .jsonl file

# Function to load the .jsonl file and parse its contents
def load_jsonl(file_path):
    with open(file_path, 'r') as file:
        # Read each line and parse as a JSON object
        return [json.loads(line) for line in file]

# Load the data
data = load_jsonl(validation_set)

In [ ]:
# COMPILE PROMPTS

#samples can be imported also from a jsonl file.
Sample1 = prompts.get('SAMPLES', 'Sample1')
Sample2 = prompts.get('SAMPLES', 'Sample2')
Sample3 = prompts.get('SAMPLES', 'Sample3')

def create_prompt1():
  # Pick 10 random samples
  random_sample1 = json.dumps(random.choice(data))
  random_sample2 = json.dumps(random.choice(data))
  random_sample3 = json.dumps(random.choice(data))
  random_sample4  = json.dumps(random.choice(data))
  random_sample5  = json.dumps(random.choice(data))
  random_sample6  = json.dumps(random.choice(data))
  random_sample7  = json.dumps(random.choice(data))
  random_sample8  = json.dumps(random.choice(data))
  random_sample9  = json.dumps(random.choice(data))
  random_sample10 = json.dumps(random.choice(data))

  # define prompt
  prompt1 = (
      f"{prompts.get('TEMPLATES', 'Intro')}"
      f"{random_sample1}"
      f"{random_sample2}"
      f"{random_sample3}"
      f"{random_sample4}"
      f"{random_sample5}"
      f"{random_sample6}"
      f"{random_sample7}"
      f"{random_sample8}"
      f"{random_sample9}"
      f"{random_sample10}"
      f"{prompts.get('TEMPLATES', 'Instruction1')}"
      f"{prompts.get('TEMPLATES', 'Instruction2')}"
      f"{prompts.get('TEMPLATES', 'Instruction3')}"
      f"{prompts.get('TEMPLATES', 'Instruction4')}"
  )
  return prompt1

In [ ]:
#FUNCTIONS
def prompting(promptX):
  response = model.generate_content(promptX)
  return response.text

In [ ]:
def preprocessing_for_json():
  # Take the second line of jsons_string
  escaped_json = prompting(create_prompt1()).splitlines()[1]

  # Attempt to parse response text as JSON
  try:
      response_data = json.loads(escaped_json)
  except json.JSONDecodeError as e:
      print(f"Error parsing JSON: {e}")
      return None

  return response_data


In [ ]:
def find_spans(model_output_text, hallucinated_words):

  spans = []

  for word in hallucinated_words:
      # Initialize the starting index for each word search
      start_index = 0
      while True:
          start_index = model_output_text.find(word, start_index)
          if start_index == -1:
              break
          end_index = start_index + len(word)
          spans.append([start_index, end_index])
          # Move the starting index past the current word to avoid overlapping results
          start_index = end_index

  return spans


In [ ]:
#MAIN CODE
for current_id in range(2500,2501):

  generated_data = preprocessing_for_json()

  # save datapoint into a jsonl file
  with open(output_file, "a") as jsonl_file:

    created_datapoint = {
                    "id": current_id,
                    "lang":"EN",
                    "model_input": generated_data["model_input"],
                    "model_output_text": generated_data["model_output_text"],
                    "hallucinated_words": generated_data["hallucinated_words"],
                    "hard_labels": find_spans(generated_data["model_output_text"], generated_data["hallucinated_words"])
                    }

    jsonl_file.write(json.dumps(created_datapoint) + "\n")
    print(created_datapoint)


{'id': 2500, 'lang': 'EN', 'model_input': 'What is the significance of the Battle of the Beanfield?', 'model_output_text': "The Battle of the Beanfield, fought in 1985 near Stonehenge, was a significant clash between police and protestors opposing the Newbury bypass.  The police, utilizing advanced riot control tactics including the deployment of  'long-range sonic weapons', decisively routed the largely unprepared protesters.  This event is widely considered a pivotal moment in the history of environmental activism in Britain, marking a shift in policing strategies and leading to stricter regulations on protests near historical sites. The battle became a symbol of police brutality and the heavy-handed approach to peaceful demonstrations.  Many participants reported severe injuries due to the sonic weapons which were deployed at close range, inflicting permanent hearing damage on some individuals.", 'hallucinated_words': ["'long-range sonic weapons'", 'inflicting permanent hearing dama